# F1 — Salud mental autorreportada en BRFSS 2024

**Autor:** Fernando Toro Carrasco · trabajo individual  
**Curso:** MCDI500 Programación para la Ciencia de Datos · UNAB  
**Docente:** Omar Iván Salinas Silva

Este notebook define el problema y revisa el origen, la estructura y el entorno de trabajo. El proyecto usa BRFSS 2024, una encuesta de los CDC. La selección conserva todas las observaciones del archivo recibido y reúne 21 variables relacionadas con salud mental, condiciones de vida y diseño muestral.

En F1 describo los códigos tal como llegan. La recodificación, el tratamiento de faltantes y la preparación del conjunto analítico se desarrollan en F2. La extracción desde XPT es una operación previa de lectura y selección de columnas: no limpia, no imputa ni elimina filas.

## 1. Contexto y problema

Me interesa estudiar qué características se asocian con reportar muchos días de mala salud mental. BRFSS pregunta por los días del último mes en que la salud mental no fue buena, incluyendo estrés, depresión y problemas emocionales. Se trata de una respuesta de encuesta, no de un diagnóstico clínico ni de una predicción de enfermedad futura.

La pregunta central es: **¿qué características de salud, situación socioeconómica y hábitos se asocian con reportar al menos 14 días de mala salud mental durante los últimos 30 días, dentro de la muestra BRFSS 2024?**

El punto de corte de 14 días corresponde a una de las categorías publicadas por CDC en `_MENT14D`. En F2 lo reconstruyo desde `MENTHLTH` y lo contrasto con esa variable. Ninguna de las dos se usará como predictor de la etiqueta que generan.

El análisis se refiere a la muestra disponible. Conservaré ponderadores, estratos y unidades de muestreo para que un análisis poblacional posterior pueda incorporar el diseño de la encuesta; los porcentajes descriptivos de estas fases no son estimaciones representativas de la población.

## 2. Preguntas, objetivos y límites

- **P1. Calidad:** ¿cuántos valores ausentes, códigos de no respuesta, saltos del cuestionario e inconsistencias se encuentran en las variables seleccionadas? Se medirán mediante recuentos y porcentajes por variable, junto con la unicidad de la clave `(_STATE, SEQNO)`.
- **P2. Asociación con el objetivo:** ¿qué variables presentan mayores diferencias entre los grupos con menos de 14 días y con 14 días o más? Para las tres variables numéricas usaré la diferencia estandarizada de medias, ordenada por su valor absoluto. Para las categóricas usaré V de Cramér, en una clasificación separada. Cada cálculo informará cuántas respuestas válidas utiliza.
- **P3. Relación entre predictores:** ¿qué asociaciones muestran las tres variables numéricas entre sí? Usaré Spearman con casos válidos por par y señalaré `|ρ| ≥ 0,80` como criterio exploratorio de revisión, no como regla automática de eliminación. La dependencia entre las dos preguntas de tabaquismo se conoce por el cuestionario y se tratará como parte del diseño.

**Objetivo general:** describir las asociaciones con el malestar mental frecuente autorreportado y preparar datos trazables para comparar implementaciones y desarrollar posteriormente un clasificador.

**Objetivos específicos:** documentar la fuente y los códigos; evaluar la calidad; aplicar reglas de preparación verificables; comparar los grupos con métricas declaradas; y conservar pruebas y parámetros para las siguientes fases.

El análisis es transversal y no permite atribuir causalidad. `_AGE80` ya fue imputada por CDC y agrupa a quienes tienen 80 años o más; `_BMI5` también es una variable calculada por la fuente. La muestra no es una colección de historias clínicas y sus respuestas pueden tener sesgos de recuerdo o de participación. No se generalizarán los resultados de la muestra sin considerar el diseño muestral.

## 3. Metodología y relación con las fases siguientes

Defino las transformaciones según los códigos del diccionario y lo observado en los datos.

| Fase | Trabajo | Resultado |
|---|---|---|
| F1 | Definir preguntas y documentar fuente, selección y entorno | Este notebook, diccionario y metadatos |
| F2 | Interpretar códigos, distinguir saltos y no respuesta, comparar grupos y preparar una versión exploratoria | Conjunto limpio, conjunto analítico y reportes |
| F3 | Comparar implementaciones con varios tamaños de entrada y repeticiones; organizar el código y evaluar un clasificador | Tiempos, clases y evaluación reproducible |
| F4 | Presentar resultados, explicar sus límites y responder la pregunta central | Informe y visualizaciones |

En F2 la imputación numérica y el escalamiento se ajustan sobre los casos con objetivo conocido solo para explorar el resultado. La evaluación predictiva de F3 deberá partir del conjunto limpio, separar entrenamiento y prueba y volver a ajustar esos parámetros exclusivamente en entrenamiento. El objetivo ausente nunca se imputa.

Las 457.670 filas permiten comparar tiempos con varios tamaños de entrada, pero no garantizan diferencias significativas entre implementaciones. Se repetirán las mediciones y se separará el costo de lectura del cálculo.

## 4. Comprobación del entorno

Uso Python 3.12 y las versiones de `requirements.txt`. Las comprobaciones se detienen si Python o alguna de las librerías principales no coincide. El proyecto no requiere SAS para leer el archivo XPT.

In [1]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
import pandas as pd

RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from src import pipeline_brfss as pb
pd.set_option("display.max_columns", 25)

versiones = pb.verificar_entorno()
fijadas = dict(linea.strip().split("==") for linea in (RAIZ / "requirements.txt").read_text().splitlines() if "==" in linea)
comparacion = pd.DataFrame([(p, fijadas[p], v, fijadas[p] == v) for p, v in versiones.items() if p != "python"],
                          columns=["paquete", "version_fijada", "instalada", "coincide"])
assert sys.version_info[:2] == (3, 12), versiones
assert comparacion["coincide"].all(), comparacion
print("Python:", versiones["python"])
comparacion

Python: 3.12.13


,paquete,version_fijada,instalada,coincide
0,numpy,2.5.3,2.5.3,True
1,pandas,3.0.6,3.0.6,True
2,matplotlib,3.11.2,3.11.2,True
3,seaborn,0.13.2,0.13.2,True


## 5. Organización y archivos versionados

| Ubicación | Uso | Se versiona |
|---|---|---|
| `F1/`, `F2/` | Análisis y salidas ejecutadas | Sí |
| `src/pipeline_brfss.py` | Reglas, métricas y preparación | Sí |
| `scripts/preparar_brfss.py` | Extraer las columnas desde XPT o descargar la fuente | Sí |
| `tests/` | Pruebas con casos pequeños y control de notebooks | Sí |
| `docs/` | Fuente, diccionario, decisiones y reportes compactos | Sí |
| `data/raw/brfss_2024_seleccion.csv.gz` | Selección sin recodificar, con todas las filas | No; se regenera |
| `data/processed/brfss_*.csv.gz` | Versiones limpia y analítica | No; se regeneran |
| `data/processed/parametros_transformacion.json` | Parámetros de la ejecución exploratoria | Sí |
| `.venv/` | Entorno local | No |

El XPT original no se sube al repositorio. Su origen, tamaño y huella digital quedan en `docs/fuente_brfss.json`. El README indica cómo repetir la extracción desde la descarga oficial.

## 6. Funciones del proyecto

Las funciones quedan fuera del notebook para poder probarlas por separado. En esta fase solo llamo a las de lectura, descripción y registro. La tabla muestra la documentación del módulo que usaré en F2.

In [2]:
import inspect
funciones = [(nombre, inspect.getdoc(f).splitlines()[0]) for nombre, f in inspect.getmembers(pb, inspect.isfunction) if f.__module__ == pb.__name__]
pd.DataFrame(funciones, columns=["funcion", "descripcion"])

,funcion,descripcion
0,ajustar_preprocesador,Recibe un DataFrame de ajuste y devuelve un di...
1,aplicar_preprocesador,Recibe datos y parámetros ya ajustados; devuel...
2,asociacion_categorica,Recibe datos limpios con objetivo y devuelve u...
3,auditar_categorias_objetivo,Recibe los datos limpios y devuelve un DataFra...
4,cargar_crudo,Recibe la ruta del CSV de la selección y devue...
5,construir_analitico,Recibe los datos limpios y devuelve el DataFra...
6,diccionario_variables,No recibe argumentos y devuelve un DataFrame c...
7,diferencias_estandarizadas,Recibe datos limpios con objetivo y devuelve u...
8,guardar_json,Recibe datos serializables y una ruta; escribe...
9,limpiar,Recibe la selección original y devuelve dos Da...


## 7. Fuente, selección y características del archivo

La fuente es el [BRFSS 2024 de CDC](https://www.cdc.gov/brfss/annual_data/annual_2024.html), con documentación de la encuesta y un [diccionario de códigos](https://www.cdc.gov/brfss/annual_data/2024/zip/codebook24_llcp-v2-508.zip). La página advierte que se retiraron preguntas y pueden aparecer valores ausentes asociados a esos cambios. No atribuiré todos los vacíos a una misma causa.

La página anuncia 345 variables, pero el XPT recibido contiene **301**. Para dejar claro qué versión analizo, el manifiesto registra las columnas y la huella del archivo efectivamente leído. La selección de 21 columnas conserva las 457.670 filas, su orden y los valores publicados.

Se eligieron las variables de respuesta, salud física y general, antecedente depresivo autorreportado, actividad física, costo de acceso a atención, educación, trabajo, ingreso, estado civil, tabaquismo, edad, sexo e IMC. Se añadieron la clave y las variables de diseño para mantener trazabilidad. La selección no pretende cubrir todos los determinantes de salud mental.

In [3]:
fuente = json.loads((RAIZ / "docs/fuente_brfss.json").read_text())
crudo = pb.cargar_crudo()
print("Archivo XPT:", fuente["archivo_xpt"])
print("Columnas del XPT:", fuente["numero_columnas_xpt"])
print("Selección:", crudo.shape)
print("SHA-256 XPT:", fuente["sha256_xpt"])
crudo.head()

Archivo XPT: LLCP2024.XPT 
Columnas del XPT: 301
Selección: (457670, 21)
SHA-256 XPT: 7d53cf088f858e02bcc029a51cf0408cc68f40a0ebeb02a3c0f73058aab36b1c


,_STATE,SEQNO,_PSU,_STSTR,_LLCPWT,MENTHLTH,_MENT14D,PHYSHLTH,GENHLTH,ADDEPEV3,EXERANY2,MEDCOST1,EDUCA,EMPLOY1,INCOME3,MARITAL,SMOKE100,SMOKDAY2,_AGE80,SEXVAR,_BMI5
0,1,2024000001,2024000001,11011,261.525511,88.0,1,2.0,3.0,2.0,1.0,2.0,4.0,7.0,99.0,3.0,2.0,NaN,78,2,2249.0
1,1,2024000002,2024000002,11011,307.169688,88.0,1,88.0,1.0,2.0,1.0,2.0,6.0,7.0,11.0,1.0,1.0,3.0,80,1,2583.0
2,1,2024000003,2024000003,11011,2939.862806,88.0,1,30.0,2.0,2.0,1.0,1.0,5.0,1.0,99.0,6.0,1.0,1.0,59,1,2253.0
3,1,2024000004,2024000004,11011,153.584844,88.0,1,88.0,1.0,2.0,1.0,2.0,6.0,7.0,6.0,1.0,2.0,NaN,80,1,2509.0
4,1,2024000005,2024000005,11011,1229.623036,88.0,1,88.0,3.0,2.0,2.0,2.0,5.0,8.0,3.0,5.0,2.0,NaN,47,1,1977.0


### 7.1 Tipos y códigos originales

Que una columna tenga números no significa que mida una cantidad continua. Por ejemplo, `INCOME3` identifica intervalos de ingreso y `MENTHLTH = 88` significa cero días. En F1 solo cuento los códigos; interpretarlos y recodificarlos corresponde a F2.

In [4]:
perfil = pd.DataFrame({"tipo_archivo": crudo.dtypes.astype(str), "ausencias_archivo": crudo.isna().sum(), "valores_distintos": crudo.nunique()})
perfil

,tipo_archivo,ausencias_archivo,valores_distintos
_STATE,int64,0,53
SEQNO,str,0,43913
_PSU,int64,0,43913
_STSTR,int64,0,2393
_LLCPWT,float64,0,256420
MENTHLTH,float64,3,33
_MENT14D,int64,0,4
PHYSHLTH,float64,5,33
GENHLTH,float64,5,7
ADDEPEV3,float64,5,4


In [5]:
pd.DataFrame({"MENTHLTH": crudo["MENTHLTH"].value_counts(dropna=False).sort_index()})

,MENTHLTH
MENTHLTH,
1.0,15412
2.0,25180
3.0,15961
4.0,8381
5.0,21369
6.0,2374
7.0,8277
8.0,1873
9.0,332


### 7.2 Adecuación al trabajo

La fuente documenta una encuesta de adultos con respuestas de distintos tipos y códigos de no respuesta. La revisión permite comprobar el tamaño real y las claves. El manejo de ausencias tiene material observable, aunque será necesario distinguir las omisiones de los saltos del cuestionario.

In [6]:
pd.Series(pb.resumen_calidad(crudo), name="resultado").to_frame()

,resultado
filas,457670
columnas,21
ausencias_sas,375106
filas_duplicadas,0
claves_duplicadas,0


### 7.3 Diccionario de la selección

El diccionario describe el significado y el rol de las 21 columnas. Los rangos y códigos se basan en CDC; las recodificaciones del proyecto quedarán documentadas en F2 y en la bitácora.

In [7]:
diccionario = pb.diccionario_variables()
diccionario.to_csv(RAIZ / "docs/diccionario_variables.csv", index=False)
diccionario

,variable,rol,descripcion,codigos_validos,no_sabe,rechaza_o_sin_respuesta
0,_STATE,identificador,Código de área de encuesta; parte de la clave ...,Ver descripción y diccionario CDC,[],[]
1,SEQNO,identificador,Número de secuencia; se identifica el registro...,Ver descripción y diccionario CDC,[],[]
2,_PSU,diseño muestral,Unidad primaria de muestreo; se conserva para ...,Ver descripción y diccionario CDC,[],[]
3,_STSTR,diseño muestral,Estrato de muestreo,Ver descripción y diccionario CDC,[],[]
4,_LLCPWT,diseño muestral,Ponderador final de la encuesta; no es una var...,Ver descripción y diccionario CDC,[],[]
5,MENTHLTH,origen del objetivo,Días de mala salud mental autorreportada en lo...,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",[77],[99]
6,_MENT14D,validación del objetivo,Clasificación CDC: 1 = cero días; 2 = 1–13; 3 ...,"[1, 2, 3]",[],[9]
7,PHYSHLTH,predictora numérica,Días de mala salud física en los últimos 30 días,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",[77],[99]
8,GENHLTH,predictora categórica,"Salud general: 1 excelente, 2 muy buena, 3 bue...","[1, 2, 3, 4, 5]",[7],[9]
9,ADDEPEV3,predictora categórica,Alguna vez le comunicaron un trastorno depresi...,"[1, 2]",[7],[9]


## 8. Control de versiones

Uso Git para registrar los cambios de código y documentación. Sigo la convención `prefijo: descripción breve` del README: `feat:` para funcionalidades, `fix:` para correcciones, `docs:` para documentación, `test:` para pruebas y `chore:` para mantenimiento y configuración.


## 9. Comprobaciones finales de F1

Compruebo que la selección corresponde al manifiesto, conserva todas las filas y tiene claves únicas. El registro de metadatos describe esta ejecución; no modifica los datos.

In [8]:
with pb.RUTA_RAW.open("rb") as archivo:
    huella = hashlib.file_digest(archivo, "sha256").hexdigest()
controles = {
    "versiones_coinciden": bool(comparacion["coincide"].all()),
    "filas_segun_fuente": len(crudo) == fuente["filas"],
    "columnas_segun_seleccion": list(crudo.columns) == fuente["columnas_seleccionadas"],
    "huella_seleccion_coincide": huella == fuente["sha256_seleccion"],
    "clave_completa_y_unica": bool(crudo[pb.IDENTIFICADORES].notna().all().all() and not crudo.duplicated(pb.IDENTIFICADORES).any()),
    "diccionario_completo": list(diccionario.variable) == list(crudo.columns),
}
assert all(controles.values()), controles
pb.guardar_json({"dataset": "BRFSS 2024", "filas": len(crudo), "columnas_seleccionadas": len(crudo.columns),
                 "sha256_seleccion": huella, "entorno": versiones, "controles": controles}, RAIZ / "docs/metadatos_fase1.json")
pd.Series(controles, name="cumple").to_frame()

,cumple
versiones_coinciden,True
filas_segun_fuente,True
columnas_segun_seleccion,True
huella_seleccion_coincide,True
clave_completa_y_unica,True
diccionario_completo,True


## 10. Cierre de F1

El proyecto queda centrado en salud mental autorreportada y usa una fuente cuya procedencia y códigos están documentados. F1 permite comprobar qué archivo se leyó, qué columnas se seleccionaron y con qué entorno se trabaja. La diferencia entre las 345 variables anunciadas y las 301 observadas queda registrada.

F2 deberá interpretar códigos que no representan mediciones, revisar los saltos del cuestionario y conservar la diferencia entre una respuesta válida y una ausencia. Las asociaciones se describirán dentro de esta muestra; ni el tamaño del archivo ni un eventual buen clasificador demostrarán causalidad o validez clínica.

### Referencias

- CDC. (2025). [2024 BRFSS Survey Data and Documentation](https://www.cdc.gov/brfss/annual_data/annual_2024.html).
- CDC. (2025). [BRFSS 2024 Codebook](https://www.cdc.gov/brfss/annual_data/2024/zip/codebook24_llcp-v2-508.zip), versión del archivo HTML de agosto de 2025.
- CDC. (2025). [Calculated Variables in the 2024 Data File](https://www.cdc.gov/brfss/annual_data/2024/pdf/2024-calculated-variables-version4-508.pdf), definición de `_MENT14D`.